# 双动量回测控制台与分析面板（合并版）

这份 notebook 用于**双动量（Dual Momentum）策略**的日常回测与结果分析，尽量保持和你现有风险平价 notebook 相同的使用习惯，但把“回测控制台”和“结果分析面板”合并成一个文件。

它会完成这些事情：

1. 读取本地市场数据  
2. 设置资产池、双动量参数与回测参数  
3. 运行双动量回测  
4. 查看关键结果与策略快照  
5. 分析净值、回撤、年度收益、月度热力图、调仓/交易统计、选中资产频率  
6. 一键导出结果

> 依赖：`market_data.py`、`backtest.py`、`dual_momentum_backtest_fixed.py`（或 `dual_momentum_backtest.py`）、`dual_momentum_fixed.py`


In [ ]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from market_data import create_manager, today_str, load_tushare_token
from backtest import build_market_matrices, calc_drawdown

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

# ========= 让当前目录可导入本地 py 文件 =========
cwd = str(Path.cwd().resolve())
if cwd not in sys.path:
    sys.path.insert(0, cwd)

# ========= 导入双动量回测模块 =========
try:
    dm_bt = importlib.import_module("dual_momentum_backtest_fixed")
    DM_BACKTEST_MODULE = "dual_momentum_backtest_fixed"
except ImportError:
    dm_bt = importlib.import_module("dual_momentum_backtest")
    DM_BACKTEST_MODULE = "dual_momentum_backtest"

simulate_dual_momentum_backtest = dm_bt.simulate_dual_momentum_backtest

print("dual momentum backtest module =", DM_BACKTEST_MODULE)

# ========= 路径与连接 =========
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"
EXPORT_DIR = Path("data/exports_dual_momentum")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("manager ready")
print("db_path =", Path(DB_PATH).resolve())
print("export_dir =", EXPORT_DIR.resolve())


In [ ]:
# ========= 资产池 =========

# 可交易风险资产池（示例：A 股宽基 / 行业 / 风格 ETF）
"""
CANDIDATE_ASSETS = [
    "510300.SH",  # 沪深300ETF
    "510500.SH",  # 中证500ETF
    "159915.SZ",  # 创业板ETF
    "588000.SH",  # 科创50ETF
    "515100.SH",  # 红利低波100ETF
    "512660.SH",  # 军工ETF
    "512480.SH",  # 半导体ETF
    "159928.SZ",  # 消费ETF
    "512170.SH",  # 医疗ETF
]
"""

CANDIDATE_ASSETS = [
    "510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    "511090.SH",  # 30年国债ETF (避险资产：超长债，对利率极度敏感)
    "159981.SZ",  # 能源ETF (抗通胀：能源/电力)
    "159985.SZ",  # 豆粕ETF (抗通胀：农产品期货)
    "501018.SH",  # 南方原油LOF (抗通胀：国际原油价格)
    "515100.SH",  # 红利低波100ETF (抗通胀：红利股)
    "513000.SH",  # 225日经ETF
    #"159659.SZ",  # 100纳斯达克ETF
]


# 防御资产（风险资产不足 top_k 时承接仓位）
DEFENSIVE_ASSETS = [
    "511010.SH",  # 国债ETF
    "518880.SH",  # 黄金ETF (抗通胀/避险：金属商品)
]

WATCHLIST = list(dict.fromkeys(CANDIDATE_ASSETS + DEFENSIVE_ASSETS))

# ========= 数据区间 =========
START_DATE = "20200101"
END_DATE = today_str()

# ========= 回测参数 =========
BACKTEST_PARAMS = {
    "initial_cash": 1_000_000.0,
    "rebalance_freq": "M",          # D / W / M / Q / Y / W-FRI
    "execution_price_type": "avg",  # open / close / high / low / avg
    "fee_rate_buy": 0.0005,
    "fee_rate_sell": 0.0005,
    "lot_size": 100,
    "max_trade_amount_ratio": 0.05,
    "amount_unit_scale": 1000.0,    # tushare 的 amount 通常按千元
    "use_drift_trigger": False,
    "drift_threshold": 0.05,
    "risk_free_rate": 0.0,          # 设为 0，则夏普更接近“收益风险比”
    "annualization": 252,
    "valuation_ffill_limit": 5,
    "execution_policy": "best_effort",  # best_effort / strict
}

# ========= 双动量预处理参数 =========
DM_PREPARE_KWARGS = {
    "ffill": True,
    "ffill_limit": 5,
    "min_non_na_ratio": 0.0,
    "drop_all_na_dates": True,
}

# ========= 双动量信号参数 =========
DM_SNAPSHOT_KWARGS = {
    "candidate_assets": CANDIDATE_ASSETS,
    "defensive_assets": DEFENSIVE_ASSETS,

    # --- 绝对动量 ---
    "abs_lookback": 60,
    "abs_threshold": 0.0,
    "abs_skip_recent": 0,
    "abs_return_type": "simple",
    "abs_ma_window": 120,          # 若不想用均线过滤，可改为 None
    "abs_ma_type": "sma",          # sma / ema

    # --- 相对动量 ---
    "rel_lookbacks": [20, 60, 120],
    "rel_weights": [0.2, 0.3, 0.5],
    "rel_skip_recent": 0,
    "rel_return_type": "simple",
    "rel_risk_adjusted": False,    # True 时会用波动率做风险调整
    "rel_vol_lookback": 20,

    # --- 组合构建 ---
    "top_k": 3,
    "weighting": "equal",          # equal / score / rank / inv_vol
    "fill_unallocated_to_defensive": True,
    "min_history": 130,
    "max_single_weight": 0.6,

    # --- 市场总开关（可选）---
    "market_asset": "510300.SH",   # 用沪深300ETF 作为市场总开关代理
    "market_lookback": 120,
    "market_threshold": 0.0,
    "market_skip_recent": 0,
    "market_ma_window": None,
    "market_ma_type": "sma",

    # --- 其他 ---
    "use_talib": False,
}

print("watchlist =", WATCHLIST)
print("candidate assets =", CANDIDATE_ASSETS)
print("defensive assets =", DEFENSIVE_ASSETS)
print("params loaded")


In [ ]:
# ========= 资产池数据完整性检查 + 自动补齐 =========
import pandas as pd
import numpy as np
from IPython.display import display

# -----------------------------
# 可调参数
# -----------------------------
CHECK_EXCHANGE = "SSE"      # 用于交易日历判断，A 股 ETF/LOF 通常用 SSE 即可
EXTRA_WARMUP_DAYS = 20      # 在策略最小 warmup 基础上再额外预留一些交易日
DO_UPDATE = True            # 只检查不更新时，改成 False
RECHECK_AFTER_UPDATE = True # 更新后是否自动复查

# -----------------------------
# 资产池
# -----------------------------
ASSET_POOL = list(dict.fromkeys(
    globals().get("WATCHLIST")
    or (globals().get("CANDIDATE_ASSETS", []) + globals().get("DEFENSIVE_ASSETS", []))
))

if len(ASSET_POOL) == 0:
    raise ValueError("未找到资产池。请先定义 WATCHLIST，或 CANDIDATE_ASSETS / DEFENSIVE_ASSETS。")

if "manager" not in globals():
    raise ValueError("未找到 manager。请先初始化 market_data manager。")

if "START_DATE" not in globals() or "END_DATE" not in globals():
    raise ValueError("未找到 START_DATE / END_DATE。请先定义回测时间区间。")

DM_KW = globals().get("DM_SNAPSHOT_KWARGS", {})

# -----------------------------
# 工具函数
# -----------------------------
def _to_ts(x):
    return pd.Timestamp(str(x))

def _yyyymmdd(ts):
    return pd.Timestamp(ts).strftime("%Y%m%d")

def _infer_required_trade_days(dm_kw: dict) -> int:
    """
    根据双动量参数，粗略推断策略运行所需的最少历史交易日长度。
    """
    if dm_kw is None:
        return 0

    need = []

    abs_lookback = int(dm_kw.get("abs_lookback", 0) or 0)
    abs_skip_recent = int(dm_kw.get("abs_skip_recent", 0) or 0)
    if abs_lookback > 0:
        need.append(abs_lookback + abs_skip_recent + 1)

    rel_lookbacks = dm_kw.get("rel_lookbacks")
    rel_lookback = dm_kw.get("rel_lookback")
    rel_skip_recent = int(dm_kw.get("rel_skip_recent", 0) or 0)

    if rel_lookbacks is not None:
        need.extend([int(x) + rel_skip_recent + 1 for x in rel_lookbacks])
    elif rel_lookback is not None:
        need.append(int(rel_lookback) + rel_skip_recent + 1)

    abs_ma_window = dm_kw.get("abs_ma_window")
    if abs_ma_window is not None:
        need.append(int(abs_ma_window))

    if bool(dm_kw.get("rel_risk_adjusted", False)):
        rel_vol_lookback = int(dm_kw.get("rel_vol_lookback", 20) or 20)
        need.append(rel_vol_lookback + 1)

    market_lookback = dm_kw.get("market_lookback")
    market_skip_recent = int(dm_kw.get("market_skip_recent", 0) or 0)
    if market_lookback is not None:
        need.append(int(market_lookback) + market_skip_recent + 1)

    market_ma_window = dm_kw.get("market_ma_window")
    if market_ma_window is not None:
        need.append(int(market_ma_window))

    min_history = dm_kw.get("min_history")
    if min_history is not None:
        need.append(int(min_history))

    return int(max(need) if len(need) > 0 else 0)


def _get_open_calendar(exchange: str, start_date: str, end_date: str) -> pd.DatetimeIndex:
    open_dates = manager.store.get_open_trade_dates(
        exchange=exchange,
        start_date=start_date,
        end_date=end_date,
    )
    if open_dates is None or len(open_dates) == 0:
        return pd.DatetimeIndex([])
    cal = pd.to_datetime(pd.Series(open_dates).astype(str), format="%Y%m%d", errors="coerce")
    cal = cal.dropna().drop_duplicates().sort_values()
    return pd.DatetimeIndex(cal)


def _infer_required_start_date(start_date: str, required_trade_days: int, exchange: str, extra_trade_days: int) -> pd.Timestamp:
    """
    根据策略所需 warmup，反推出数据库至少应覆盖到的开始日期。
    """
    start_ts = _to_ts(start_date)

    if required_trade_days <= 0 and extra_trade_days <= 0:
        return start_ts

    # 粗放地往前多取一些自然日，然后用交易日历反推
    approx_back_days = max(120, int((required_trade_days + extra_trade_days) * 2.4))
    cal_start = _yyyymmdd(start_ts - pd.Timedelta(days=approx_back_days))
    cal_end = _yyyymmdd(start_ts)

    cal = _get_open_calendar(exchange=exchange, start_date=cal_start, end_date=cal_end)
    cal = cal[cal <= start_ts]

    if len(cal) == 0:
        return start_ts

    need_back = required_trade_days + extra_trade_days
    idx = max(0, len(cal) - 1 - need_back)
    return pd.Timestamp(cal[idx])


def _infer_expected_latest_trade_date(end_date: str, exchange: str) -> pd.Timestamp:
    """
    取 <= END_DATE 的最后一个开市日，作为数据库应更新到的目标日期。
    """
    end_ts = _to_ts(end_date)
    cal_start = _yyyymmdd(end_ts - pd.Timedelta(days=40))
    cal_end = _yyyymmdd(end_ts)
    cal = _get_open_calendar(exchange=exchange, start_date=cal_start, end_date=cal_end)
    if len(cal) == 0:
        return end_ts
    return pd.Timestamp(cal.max())


def _try_get_list_date_map(ts_codes):
    """
    尽量读取 instrument 的 list_date，避免把上市较晚的标的误判为“起始不足”。
    """
    try:
        inst = manager.store.get_instruments(listed_only=False)
        if inst is None or len(inst) == 0:
            return {}
        required_cols = {"ts_code", "list_date"}
        if not required_cols.issubset(inst.columns):
            return {}
        tmp = inst.loc[inst["ts_code"].isin(ts_codes), ["ts_code", "list_date"]].copy()
        tmp["list_date"] = pd.to_datetime(tmp["list_date"].astype(str), format="%Y%m%d", errors="coerce")
        return tmp.set_index("ts_code")["list_date"].to_dict()
    except Exception as e:
        print(f"[提示] 读取 instruments/list_date 失败，起始覆盖检查将不考虑上市日期：{e}")
        return {}


def _build_integrity_report(ts_codes, required_start_ts, expected_latest_ts, exchange):
    """
    构建完整性检查报表。
    """
    query_start = _yyyymmdd(required_start_ts)
    query_end = _yyyymmdd(expected_latest_ts)

    existing = manager.store.get_daily_prices(
        ts_codes=ts_codes,
        start_date=query_start,
        end_date=query_end,
    )

    if existing is None or len(existing) == 0:
        existing = pd.DataFrame(columns=["trade_date", "ts_code"])

    grp = (
        existing.groupby("ts_code")["trade_date"]
        .agg(db_start="min", db_end="max", actual_rows="nunique")
        .reset_index()
    )

    report = pd.DataFrame({"ts_code": ts_codes}).merge(grp, on="ts_code", how="left")

    report["db_start"] = pd.to_datetime(report["db_start"].astype(str), format="%Y%m%d", errors="coerce")
    report["db_end"] = pd.to_datetime(report["db_end"].astype(str), format="%Y%m%d", errors="coerce")
    report["actual_rows"] = report["actual_rows"].fillna(0).astype(int)

    list_date_map = _try_get_list_date_map(ts_codes)

    open_calendar = _get_open_calendar(
        exchange=exchange,
        start_date=_yyyymmdd(required_start_ts),
        end_date=_yyyymmdd(expected_latest_ts),
    )

    if len(open_calendar) == 0:
        raise ValueError("交易日历为空，无法检查数据完整性。")

    def _effective_required_start(code):
        ld = list_date_map.get(code, pd.NaT)
        if pd.isna(ld):
            return required_start_ts
        return max(required_start_ts, pd.Timestamp(ld))

    report["effective_required_start"] = report["ts_code"].map(_effective_required_start)
    report["expected_latest"] = expected_latest_ts

    def _count_expected_rows(start_ts):
        return int((open_calendar >= pd.Timestamp(start_ts)).sum())

    report["expected_rows"] = report["effective_required_start"].apply(_count_expected_rows)

    report["has_data"] = (report["actual_rows"] > 0).astype(int)
    report["start_ok"] = report["db_start"].notna() & (report["db_start"] <= report["effective_required_start"])
    report["end_ok"] = report["db_end"].notna() & (report["db_end"] >= report["expected_latest"])

    # interior_ok：只要不是 0 行，且行数达到预期交易日数，就认为区间内部基本完整
    # 若存在停牌/长期不交易，也会被标红，这正适合回测前检查
    report["interior_ok"] = (report["actual_rows"] >= report["expected_rows"]) & (report["actual_rows"] > 0)

    report["needs_update"] = ~(report["start_ok"] & report["end_ok"] & report["interior_ok"])

    def _reason(row):
        reasons = []
        if row["has_data"] == 0:
            reasons.append("无数据")
        else:
            if not row["start_ok"]:
                reasons.append("起始不足")
            if not row["end_ok"]:
                reasons.append("未更新到目标结束日")
            if not row["interior_ok"]:
                reasons.append("区间内可能有缺口")
        return "；".join(reasons) if reasons else "OK"

    report["reason"] = report.apply(_reason, axis=1)

    report = report.sort_values(
        by=["needs_update", "reason", "ts_code"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    return report, existing


# -----------------------------
# 推断本次回测所需的数据时间范围
# -----------------------------
required_trade_days = _infer_required_trade_days(DM_KW)
required_start_ts = _infer_required_start_date(
    start_date=START_DATE,
    required_trade_days=required_trade_days,
    exchange=CHECK_EXCHANGE,
    extra_trade_days=EXTRA_WARMUP_DAYS,
)
expected_latest_ts = _infer_expected_latest_trade_date(
    end_date=END_DATE,
    exchange=CHECK_EXCHANGE,
)

print("资产池数量：", len(ASSET_POOL))
print("回测设置 START_DATE：", START_DATE)
print("回测设置 END_DATE：", END_DATE)
print("根据双动量参数推断所需最少 warmup 交易日：", required_trade_days)
print("数据库建议覆盖起点：", required_start_ts.strftime("%Y-%m-%d"))
print("数据库应至少更新到：", expected_latest_ts.strftime("%Y-%m-%d"))

# -----------------------------
# 首次检查
# -----------------------------
report_before, existing_before = _build_integrity_report(
    ts_codes=ASSET_POOL,
    required_start_ts=required_start_ts,
    expected_latest_ts=expected_latest_ts,
    exchange=CHECK_EXCHANGE,
)

print("\n[更新前检查结果]")
display(report_before)

to_update = report_before.loc[report_before["needs_update"], "ts_code"].tolist()

if len(to_update) == 0:
    print("\n数据库已满足当前双动量回测要求，无需更新。")
else:
    print("\n需要补齐的标的：", to_update)

    if DO_UPDATE:
        print("\n开始自动补齐数据 ...")
        update_summary = manager.update_daily_prices(
            ts_codes=to_update,
            start_date=_yyyymmdd(required_start_ts),
            end_date=_yyyymmdd(expected_latest_ts),
        )
        print("\n[自动补齐结果]")
        display(update_summary)

        # 可选：对需要补齐的资产再刷新最近 30 天，增强“最新状态”可靠性
        try:
            recent_summary = manager.refresh_recent_daily_prices(
                ts_codes=to_update,
                lookback_days=30,
                end_date=_yyyymmdd(expected_latest_ts),
            )
            print("\n[最近 30 天刷新结果]")
            display(recent_summary)
        except Exception as e:
            print(f"[提示] refresh_recent_daily_prices 执行失败，已跳过：{e}")

        if RECHECK_AFTER_UPDATE:
            report_after, existing_after = _build_integrity_report(
                ts_codes=ASSET_POOL,
                required_start_ts=required_start_ts,
                expected_latest_ts=expected_latest_ts,
                exchange=CHECK_EXCHANGE,
            )
            print("\n[更新后复查结果]")
            display(report_after)

            unresolved = report_after.loc[report_after["needs_update"], "ts_code"].tolist()
            if len(unresolved) == 0:
                print("\n所有资产已满足当前回测要求。")
            else:
                print("\n以下标的更新后仍未完全满足要求，请重点检查：", unresolved)
    else:
        print("\nDO_UPDATE=False，仅完成检查，未执行自动更新。")

In [ ]:
# ========= 读取市场数据并构建 market 字典 =========
raw_prices = manager.store.get_daily_prices(
    ts_codes=WATCHLIST,
    start_date=START_DATE,
    end_date=END_DATE,
)

print("raw_prices shape =", raw_prices.shape)
display(raw_prices.head())

coverage = raw_prices.groupby("ts_code")["trade_date"].agg(["min", "max", "count"])
display(coverage)

market = build_market_matrices(
    data=raw_prices,
    fields=("open", "high", "low", "close", "amount"),
    date_col="trade_date",
    code_col="ts_code",
    date_format="%Y%m%d",
)

DM_PREPARE_KWARGS["calendar"] = market["close"].index

for k, v in market.items():
    print(k, v.shape)


In [ ]:
# ========= 运行回测 =========
result = simulate_dual_momentum_backtest(
    market=market,
    initial_cash=BACKTEST_PARAMS["initial_cash"],
    rebalance_freq=BACKTEST_PARAMS["rebalance_freq"],
    execution_price_type=BACKTEST_PARAMS["execution_price_type"],
    valuation_ffill_limit=BACKTEST_PARAMS["valuation_ffill_limit"],
    fee_rate_buy=BACKTEST_PARAMS["fee_rate_buy"],
    fee_rate_sell=BACKTEST_PARAMS["fee_rate_sell"],
    lot_size=BACKTEST_PARAMS["lot_size"],
    max_trade_amount_ratio=BACKTEST_PARAMS["max_trade_amount_ratio"],
    amount_unit_scale=BACKTEST_PARAMS["amount_unit_scale"],
    use_drift_trigger=BACKTEST_PARAMS["use_drift_trigger"],
    drift_threshold=BACKTEST_PARAMS["drift_threshold"],
    dm_prepare_kwargs=DM_PREPARE_KWARGS,
    dm_snapshot_kwargs=DM_SNAPSHOT_KWARGS,
    risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
    annualization=BACKTEST_PARAMS["annualization"],
    include_cash_column=True,
    cash_column_name="CASH",
    store_snapshot_details=True,
    execution_policy=BACKTEST_PARAMS["execution_policy"],
)

summary = result["summary"]
nav_df = result["nav_df"]
returns = result["returns"]
weights_df = result["weights_df"]
positions_df = result["positions_df"]
target_weights_df = result["target_weights_df"]
trades_df = result["trades_df"]
rebalance_log_df = result["rebalance_log_df"]
snapshot_log_df = result["snapshot_log_df"]
snapshot_details = result.get("snapshot_details", {})

print("回测完成")
display(summary.to_frame("value"))


In [ ]:
# ========= 日常查看：净值 / 回撤 / 现金占比 / 最近调仓 / 最近交易 / 最新策略快照 =========
nav = nav_df["nav"].copy()
drawdown = calc_drawdown(nav)
cash_weight = nav_df["cash_weight"].copy() if "cash_weight" in nav_df.columns else (nav_df["cash"] / nav_df["nav"])

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# 1. 净值
axes[0].plot(nav_df.index, nav, label="NAV")
axes[0].set_title("Portfolio NAV")
axes[0].legend()

# 2. 回撤
axes[1].plot(drawdown.index, drawdown, label="Drawdown")
axes[1].set_title("Drawdown")
axes[1].legend()

# 3. 现金占比
axes[2].plot(cash_weight.index, cash_weight, label="Cash Weight")
axes[2].set_title("Cash Weight")
axes[2].legend()

plt.tight_layout()
plt.show()

print("最近净值：")
display(nav_df.tail())

print("最近现金占比：")
display(cash_weight.tail().to_frame("cash_weight"))

print("最近目标权重：")
display(target_weights_df.tail())

print("最近实际权重：")
display(weights_df.tail())

print("最近持仓：")
display(positions_df.tail())

print("最近调仓日志：")
display(rebalance_log_df.tail(10))

print("最近交易记录：")
display(trades_df.tail(20))

print("最近信号日志：")
display(snapshot_log_df.tail(10))

if snapshot_details:
    last_signal_date = max(snapshot_details.keys())
    print(f"最近一次 snapshot 细节：{last_signal_date}")
    display(snapshot_details[last_signal_date])

print("平均现金占比：", round(float(cash_weight.mean()), 4))
print("最大现金占比：", round(float(cash_weight.max()), 4))


In [ ]:
# ========= 资产收益相关性矩阵（基于 close 收益率） + 实际持有比例变化 =========
close_px = market["close"].copy()
asset_ret = close_px.pct_change().dropna(how="all")
asset_corr_matrix = asset_ret[CANDIDATE_ASSETS + DEFENSIVE_ASSETS].corr()

try:
    import seaborn as sns
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        asset_corr_matrix,
        annot=True,
        cmap="coolwarm",
        vmin=-1, vmax=1,
        center=0,
        fmt=".2f",
        linewidths=1,
        square=True,
    )
    plt.title("Asset Correlation Matrix", fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()
except Exception:
    display(asset_corr_matrix)

display(asset_corr_matrix)

actual_weights_plot = weights_df.copy().fillna(0.0)

plt.figure(figsize=(14, 6))
plt.stackplot(
    actual_weights_plot.index,
    actual_weights_plot.T.values,
    labels=actual_weights_plot.columns
)
plt.title("Actual Portfolio Weights Over Time")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()


In [ ]:
# ========= 双动量专属分析：选中资产数量 / 目标现金 / 资产入选频率 / 最新明细 =========
if len(snapshot_log_df) > 0:
    tmp = snapshot_log_df.copy()
    tmp["signal_date"] = pd.to_datetime(tmp["signal_date"])
    tmp = tmp.set_index("signal_date").sort_index()

    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    axes[0].plot(tmp.index, tmp["selected_count"], label="Selected Count")
    axes[0].set_title("Selected Asset Count")
    axes[0].legend()

    axes[1].plot(tmp.index, tmp["target_cash_weight"], label="Target Cash Weight")
    axes[1].set_title("Target Cash Weight")
    axes[1].legend()

    market_regime_num = tmp["market_regime_on"].astype(int)
    axes[2].step(tmp.index, market_regime_num, where="post", label="Market Regime On")
    axes[2].set_title("Market Regime Filter (1=On, 0=Off)")
    axes[2].set_ylim(-0.05, 1.05)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    # 统计各资产被选中的次数
    selected_pairs = []
    for dt, s in tmp["selected_assets"].fillna("").items():
        codes = [x.strip() for x in str(s).split(",") if x.strip()]
        for code in codes:
            selected_pairs.append((dt, code))

    if len(selected_pairs) > 0:
        selected_df = pd.DataFrame(selected_pairs, columns=["signal_date", "ts_code"])
        selected_freq = selected_df["ts_code"].value_counts().rename("selected_count").to_frame()
        selected_freq["selected_ratio"] = selected_freq["selected_count"] / len(tmp)
        print("资产入选频率：")
        display(selected_freq)

        plt.figure(figsize=(10, 4))
        selected_freq["selected_ratio"].sort_values(ascending=False).plot(kind="bar")
        plt.title("Selected Asset Frequency")
        plt.ylabel("Ratio")
        plt.tight_layout()
        plt.show()
    else:
        print("snapshot_log_df 中没有 selected_assets 记录")

    if snapshot_details:
        last_signal_date = max(snapshot_details.keys())
        detail = snapshot_details[last_signal_date].copy()
        if {"selected", "rel_score"}.issubset(detail.columns):
            detail = detail.sort_values(["selected", "rel_score"], ascending=[False, False])
        print(f"最近一次 snapshot 明细（{last_signal_date}）：")
        display(detail)
else:
    print("没有 snapshot_log_df，无法做双动量信号分析")


In [ ]:
# ========= 年度收益 / 月度收益热力图 =========
year_end_nav = nav.resample("Y").last()
annual_returns = year_end_nav.pct_change()
annual_returns.index = annual_returns.index.year

annual_returns_df = annual_returns.to_frame("annual_return")
display(annual_returns_df)

plt.figure(figsize=(10, 4))
plt.bar(annual_returns_df.index.astype(str), annual_returns_df["annual_return"])
plt.title("Annual Returns")
plt.show()

month_end_nav = nav.resample("M").last()
monthly_returns = month_end_nav.pct_change()
monthly_df = monthly_returns.to_frame("monthly_return")
monthly_df["year"] = monthly_df.index.year
monthly_df["month"] = monthly_df.index.month

heatmap_table = monthly_df.pivot(index="year", columns="month", values="monthly_return")
display(heatmap_table.style.format("{:.2%}"))

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(heatmap_table.values, aspect="auto")

ax.set_xticks(range(len(heatmap_table.columns)))
ax.set_xticklabels(heatmap_table.columns)
ax.set_yticks(range(len(heatmap_table.index)))
ax.set_yticklabels(heatmap_table.index)

ax.set_title("Monthly Return Heatmap")
ax.set_xlabel("Month")
ax.set_ylabel("Year")

for i in range(heatmap_table.shape[0]):
    for j in range(heatmap_table.shape[1]):
        val = heatmap_table.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.1%}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# ========= 调仓统计 / 交易统计 / 权重分析 / 最近持仓与目标权重对比 =========
if len(rebalance_log_df) > 0:
    rebalance_log_df["trade_date"] = pd.to_datetime(rebalance_log_df["trade_date"])
    rebalance_log_df["year"] = rebalance_log_df["trade_date"].dt.year

    print("调仓日志：")
    display(rebalance_log_df.tail(20))

    yearly_rebalance_count = rebalance_log_df.groupby("year").size().rename("rebalance_count")
    yearly_turnover = rebalance_log_df.groupby("year")["turnover"].sum().rename("turnover_sum")

    extra_cols = [c for c in ["partial_execution", "signal_cleared"] if c in rebalance_log_df.columns]
    agg_frames = [yearly_rebalance_count, yearly_turnover]
    if "partial_execution" in rebalance_log_df.columns:
        yearly_partial = rebalance_log_df.groupby("year")["partial_execution"].sum().rename("partial_execution_count")
        agg_frames.append(yearly_partial)

    display(pd.concat(agg_frames, axis=1))
else:
    print("没有调仓记录")

if len(trades_df) > 0:
    trade_summary = trades_df.groupby(["ts_code", "side"])["trade_value"].agg(["count", "sum"])
    display(trade_summary)

    fee_summary = trades_df.groupby("side")["cost"].sum().rename("total_cost")
    display(fee_summary)
else:
    print("没有交易记录")

print("最后一期实际权重：")
display(
    weights_df.tail(1).T.rename(columns={weights_df.index[-1]: "last_weight"})
    .sort_values("last_weight", ascending=False)
)

print("样本期平均实际权重：")
avg_weight = weights_df.mean().sort_values(ascending=False).to_frame("avg_weight")
display(avg_weight)

plt.figure(figsize=(10, 4))
avg_weight["avg_weight"].plot(kind="bar")
plt.title("Average Actual Weights")
plt.show()

if len(weights_df) > 0 and len(target_weights_df) > 0:
    last_actual = weights_df.tail(1).T.rename(columns={weights_df.index[-1]: "actual_weight"})
    last_target = target_weights_df.tail(1).T.rename(columns={target_weights_df.index[-1]: "target_weight"})
    compare = last_actual.join(last_target, how="outer").fillna(0.0)
    compare["diff"] = compare["actual_weight"] - compare["target_weight"]
    print("最近实际权重 vs 目标权重：")
    display(compare.sort_values("target_weight", ascending=False))

print("最近持仓：")
display(positions_df.tail(1).T.rename(columns={positions_df.index[-1]: "shares"}).sort_values("shares", ascending=False))


In [ ]:
# ========= 一键导出 =========
nav_df.to_csv(EXPORT_DIR / "nav.csv")
weights_df.to_csv(EXPORT_DIR / "weights.csv")
positions_df.to_csv(EXPORT_DIR / "positions.csv")
target_weights_df.to_csv(EXPORT_DIR / "target_weights.csv")
trades_df.to_csv(EXPORT_DIR / "trades.csv", index=False)
rebalance_log_df.to_csv(EXPORT_DIR / "rebalance_log.csv", index=False)
snapshot_log_df.to_csv(EXPORT_DIR / "snapshot_log.csv", index=False)
summary.to_csv(EXPORT_DIR / "summary.csv")
asset_corr_matrix.to_csv(EXPORT_DIR / "asset_corr_matrix.csv")

if snapshot_details:
    snapshot_detail_dir = EXPORT_DIR / "snapshot_details"
    snapshot_detail_dir.mkdir(parents=True, exist_ok=True)
    for dt, df in snapshot_details.items():
        dt_str = pd.Timestamp(dt).strftime("%Y%m%d")
        df.to_csv(snapshot_detail_dir / f"snapshot_detail_{dt_str}.csv")

print("export done")


In [ ]:
# ========= 可选：快速改参数重跑 =========
# 这里只留几个最常用示例，避免 notebook 过长

# BACKTEST_PARAMS["rebalance_freq"] = "W-FRI"
# BACKTEST_PARAMS["use_drift_trigger"] = True
# BACKTEST_PARAMS["drift_threshold"] = 0.08

# DM_SNAPSHOT_KWARGS["top_k"] = 1
# DM_SNAPSHOT_KWARGS["weighting"] = "score"
# DM_SNAPSHOT_KWARGS["rel_risk_adjusted"] = True
# DM_SNAPSHOT_KWARGS["fill_unallocated_to_defensive"] = True
# DM_SNAPSHOT_KWARGS["abs_ma_window"] = None
# DM_SNAPSHOT_KWARGS["market_asset"] = None

# result = simulate_dual_momentum_backtest(
#     market=market,
#     initial_cash=BACKTEST_PARAMS["initial_cash"],
#     rebalance_freq=BACKTEST_PARAMS["rebalance_freq"],
#     execution_price_type=BACKTEST_PARAMS["execution_price_type"],
#     valuation_ffill_limit=BACKTEST_PARAMS["valuation_ffill_limit"],
#     fee_rate_buy=BACKTEST_PARAMS["fee_rate_buy"],
#     fee_rate_sell=BACKTEST_PARAMS["fee_rate_sell"],
#     lot_size=BACKTEST_PARAMS["lot_size"],
#     max_trade_amount_ratio=BACKTEST_PARAMS["max_trade_amount_ratio"],
#     amount_unit_scale=BACKTEST_PARAMS["amount_unit_scale"],
#     use_drift_trigger=BACKTEST_PARAMS["use_drift_trigger"],
#     drift_threshold=BACKTEST_PARAMS["drift_threshold"],
#     dm_prepare_kwargs=DM_PREPARE_KWARGS,
#     dm_snapshot_kwargs=DM_SNAPSHOT_KWARGS,
#     risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
#     annualization=BACKTEST_PARAMS["annualization"],
#     include_cash_column=True,
#     cash_column_name="CASH",
#     store_snapshot_details=True,
#     execution_policy=BACKTEST_PARAMS["execution_policy"],
# )
